# 🔴 Day 11 — Map-Reduce with `Send` (Orchestrator–Workers)

**LangGraph from Zero to Advanced · Day 11 of 16**

**Today's goal:** run a node **many times in parallel** when you only know *at runtime* how many — e.g. "summarize each of these N documents" or "write each section of a report".

| Lesson | Topic |
|---|---|
| 1 | The problem: unknown number of parallel tasks |
| 2 | `Send` basics (no LLM) |
| 3 | Map-reduce with an LLM: summarize many reviews |
| 4 | Orchestrator–workers: a report writer |
| 5 | Practice |

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — The problem

On Day 10 we drew 3 parallel edges by hand. But what if the number of tasks comes from data?

* 7 uploaded files → 7 summaries
* the LLM plans 4 report sections → 4 writers

You can't draw edges for a number you don't know yet. **`Send`** solves this:

```python
Send("node_name", {"input": "for this one copy"})
```

A router returns a **list of `Send`s** → LangGraph runs one copy of the node **per `Send`**, all in parallel, each with its **own input**. Results are merged with a reducer. This is **map-reduce**:

```
            ┌► worker(item 1) ┐
START ► plan┼► worker(item 2) ┼► reduce ► END     (number of workers decided at runtime)
            └► worker(item N) ┘
```

## Lesson 2 — `Send` basics (no LLM)

Square every number in a list — one worker per number.

In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class MainState(TypedDict):
    numbers: list[int]
    squares: Annotated[list[int], operator.add]    # reducer collects worker results
    total: int

class WorkerInput(TypedDict):                      # what ONE worker receives
    n: int

def fan_out(state: MainState):
    # 👇 router: one Send per number
    return [Send("square", {"n": n}) for n in state["numbers"]]

def square(inp: WorkerInput) -> dict:
    print(f"  worker squaring {inp['n']}")
    return {"squares": [inp["n"] ** 2]}           # writes into the MAIN state

def add_up(state: MainState) -> dict:
    return {"total": sum(state["squares"])}

g = StateGraph(MainState)
g.add_node("square", square)
g.add_node("add_up", add_up)
g.add_conditional_edges(START, fan_out, ["square"])
g.add_edge("square", "add_up")
g.add_edge("add_up", END)
squares_app = g.compile()
show_graph(squares_app)

res = squares_app.invoke({"numbers": [1, 2, 3, 4, 5]})
print(res)
assert sorted(res["squares"]) == [1, 4, 9, 16, 25] and res["total"] == 55
print("✅ 5 workers ran in parallel!")

🔑 **Key points**

* The worker gets **only** what you put in `Send(...)`, not the whole state.
* The worker's return value updates the **main** state → use a reducer (`operator.add`) to collect.
* `add_up` runs once, after **all** workers finish.

## Lesson 3 — Map-reduce with an LLM: summarize many reviews

Each review is analysed in parallel (**map**), then one node writes the overall summary (**reduce**).

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class ReviewResult(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    issue: str = Field(description="main complaint or praise in max 6 words")

analyzer = llm.with_structured_output(ReviewResult)

class ReviewsState(TypedDict):
    reviews: list[str]
    results: Annotated[list[dict], operator.add]
    summary: str

def send_reviews(state: ReviewsState):
    return [Send("analyze", {"review": r}) for r in state["reviews"]]

def analyze(inp: dict) -> dict:
    r = analyzer.invoke(f"Analyze this product review: {inp['review']}")
    return {"results": [{"review": inp["review"], **r.model_dump()}]}

def summarize(state: ReviewsState) -> dict:
    lines = "\n".join(f"- {x['sentiment']}: {x['issue']}" for x in state["results"])
    text = llm.invoke(f"Here are review analyses:\n{lines}\nWrite a 3-sentence summary for the product manager.").content
    return {"summary": text}

g = StateGraph(ReviewsState)
g.add_node("analyze", analyze)
g.add_node("summarize", summarize)
g.add_conditional_edges(START, send_reviews, ["analyze"])
g.add_edge("analyze", "summarize")
g.add_edge("summarize", END)
review_app = g.compile()

reviews = [
    "Battery lasts two full days, love it!",
    "Screen cracked after one week, very poor quality.",
    "Camera is decent for the price.",
    "Delivery was late and the box was damaged.",
    "Super fast charging, great value.",
]
out = review_app.invoke({"reviews": reviews})
for r in out["results"]:
    print(f"{r['sentiment']:8} | {r['issue']}")
print("\n📋", out["summary"])

## Lesson 4 — Orchestrator–workers: a report writer

1. **Orchestrator** (LLM + structured output) plans the sections.
2. **Workers** write each section in parallel (`Send`).
3. **Synthesizer** joins them into the final report.

```
START ► orchestrator ─Send×N─► write_section ► synthesizer ► END
```

In [ ]:
class Section(BaseModel):
    title: str = Field(description="section heading")
    brief: str = Field(description="one sentence on what this section covers")

class Plan(BaseModel):
    sections: list[Section] = Field(description="3 to 4 sections")

planner = llm.with_structured_output(Plan)

class ReportState(TypedDict):
    topic: str
    sections: list[Section]
    written: Annotated[list[str], operator.add]
    report: str

def orchestrator(state: ReportState) -> dict:
    plan = planner.invoke(f"Plan a short report on: {state['topic']}")
    print("🗂 planned:", [s.title for s in plan.sections])
    return {"sections": plan.sections}

def assign_workers(state: ReportState):
    return [Send("write_section", {"section": s, "topic": state["topic"]}) for s in state["sections"]]

def write_section(inp: dict) -> dict:
    s = inp["section"]
    body = llm.invoke(f"Report topic: {inp['topic']}\nWrite the section '{s.title}' ({s.brief}). "
                      "Max 70 words, no heading.").content
    return {"written": [f"## {s.title}\n{body}"]}

def synthesizer(state: ReportState) -> dict:
    # keep the planned order (parallel workers can finish in any order)
    order = {s.title: i for i, s in enumerate(state["sections"])}
    parts = sorted(state["written"], key=lambda t: order.get(t.split("\n")[0][3:], 99))
    return {"report": f"# {state['topic']}\n\n" + "\n\n".join(parts)}

g = StateGraph(ReportState)
g.add_node("orchestrator", orchestrator)
g.add_node("write_section", write_section)
g.add_node("synthesizer", synthesizer)
g.add_edge(START, "orchestrator")
g.add_conditional_edges("orchestrator", assign_workers, ["write_section"])
g.add_edge("write_section", "synthesizer")
g.add_edge("synthesizer", END)
report_app = g.compile()
show_graph(report_app)

In [ ]:
final = report_app.invoke({"topic": "How UPI changed payments in India"})
print(final["report"])

💡 You can also send work from inside a node with `Command(goto=[Send(...), Send(...)])`.

## Lesson 5 — Practice

**Exercise 1 (no LLM).** Given a list of words, use `Send` to count the letters of each word in parallel, then find the longest word.

**Exercise 2.** Given a list of 3 cities, use `Send` to ask the LLM for one must-try food in each city, then combine them into a "food trip" list.

In [ ]:
# ✅ Solution 1
class WordsState(TypedDict):
    words: list[str]
    lengths: Annotated[list[tuple], operator.add]
    longest: str

def count_letters(inp):
    return {"lengths": [(inp["word"], len(inp["word"]))]}

g = StateGraph(WordsState)
g.add_node("count_letters", count_letters)
g.add_node("pick", lambda s: {"longest": max(s["lengths"], key=lambda x: x[1])[0]})
g.add_conditional_edges(START, lambda s: [Send("count_letters", {"word": w}) for w in s["words"]], ["count_letters"])
g.add_edge("count_letters", "pick")
g.add_edge("pick", END)
res = g.compile().invoke({"words": ["graph", "langgraph", "node", "edge"]})
print(res["longest"])
assert res["longest"] == "langgraph"

In [ ]:
# ✅ Solution 2
class FoodState(TypedDict):
    cities: list[str]
    foods: Annotated[list[str], operator.add]

def find_food(inp):
    food = llm.invoke(f"Name ONE must-try street food in {inp['city']}. Reply in max 5 words.").content
    return {"foods": [f"{inp['city']}: {food}"]}

g = StateGraph(FoodState)
g.add_node("find_food", find_food)
g.add_conditional_edges(START, lambda s: [Send("find_food", {"city": c}) for c in s["cities"]], ["find_food"])
g.add_edge("find_food", END)
for line in g.compile().invoke({"cities": ["Mumbai", "Kolkata", "Lucknow"]})["foods"]:
    print("🍲", line)
print("✅ All exercises solved!")

## 🎯 Day 11 Recap

* A router can return `[Send("node", input), ...]` → one parallel run per `Send`.
* Workers get their **own** input; they write back to the main state through a **reducer**.
* **Map-reduce:** split → process in parallel → combine.
* **Orchestrator–workers:** an LLM plans the tasks, workers do them, a synthesizer joins them.

**Tomorrow (Day 12):** **subgraphs** and **multi-agent** systems.